# Macro data demo
MacroSeries, MacroReleases and FREDFetcher: validation with made-up data, then real data from FRED/ALFRED.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd

from models.fred_fetcher import FREDFetcher
from models.macro_releases import MacroReleases
from models.macro_series import MacroSeries
from models.wb_fetcher import WorldBankFetcher

## MacroSeries: one version of a series
A validated time series with standard metadata. Below: a valid case and the errors it rejects.

In [2]:
META = {
    'series_id': 'GDP', 'name': "Gross Domestic Product", 'source': 'BCE',
    'frequency': 'Q', 'unit': '€', 'country': 'ES', 'category': 'Economy',
}

dates = pd.date_range('2024-01-01', periods=4, freq='QS')
good = pd.Series([2.0, 3.5, np.nan, 10.0], index=dates[[2, 0, 1, 3]])

m = MacroSeries(data=good, **META)
print(m)
print(m.data)

MacroSeries(series_id='GDP', name='Gross Domestic Product', source='BCE', frequency=<Frequency.QUARTERLY: 'Q'>, unit='€', country='ES', category='Economy', release='latest', seasonally_adjusted=None, mean=5.166666666666667, std=4.252450274057691, start=Timestamp('2024-01-01 00:00:00'), end=Timestamp('2024-10-01 00:00:00'), n_obs=3)
2024-01-01     3.5
2024-04-01     NaN
2024-07-01     2.0
2024-10-01    10.0
dtype: float64


In [3]:
bad_cases = {
    "not a Series": [1, 2, 3],
    "index not dates": pd.Series([2, 3.5, 10]),
    "empty": pd.Series([], index=pd.DatetimeIndex([]), dtype=float),
    "text values": pd.Series(['a', 'b', 'c'], index=dates[:3]),
}
for label, data in bad_cases.items():
    try:
        MacroSeries(data=data, **META)
        print(f"{label}: NO ERROR (wrong)")
    except (TypeError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

not a Series: TypeError: data must be a Series, got list
index not dates: ValueError: index must contain dates, got dtype int64
empty: ValueError: data must not be empty, got 0 rows
text values: ValueError: data values must be numeric, got dtype str


## MacroReleases: all published versions
ALFRED keeps every version of a macro series as it was published. MacroReleases stores them and rebuilds the series as first published, as last revised, or as known on a given date. Like ALFRED, a new row appears only when a value changes, so release 2 is the first revision. Data below is made up: January to April, with a late revision in February.

In [4]:
vintages = pd.DataFrame(
    [
        # date (reference month), release_date (publication), value
        ('2024-01-01', '2024-02-02', 3.7),
        ('2024-01-01', '2024-03-08', 3.8),
        ('2024-02-01', '2024-03-08', 3.9),
        ('2024-02-01', '2024-06-07', 4.0),  # late revision
        ('2024-03-01', '2024-04-05', 3.8),
        ('2024-03-01', '2024-05-03', 3.7),
        ('2024-04-01', '2024-05-03', 3.9),
        ('2024-04-01', '2024-06-07', 4.0),
    ],
    columns=['date', 'release_date', 'value'],
)
vintages['date'] = pd.to_datetime(vintages['date'])
vintages['release_date'] = pd.to_datetime(vintages['release_date'])
vintages

,date,release_date,value
0,2024-01-01,2024-02-02,3.7
1,2024-01-01,2024-03-08,3.8
2,2024-02-01,2024-03-08,3.9
3,2024-02-01,2024-06-07,4.0
4,2024-03-01,2024-04-05,3.8
5,2024-03-01,2024-05-03,3.7
6,2024-04-01,2024-05-03,3.9
7,2024-04-01,2024-06-07,4.0


In [5]:
RELEASES_META = {
    'series_id': 'UNRATE', 'name': "Unemployment Rate", 'source': 'FRED',
    'frequency': 'M', 'unit': '%', 'country': 'US', 'category': 'Labor',
}
releases = MacroReleases(data=vintages, **RELEASES_META)
releases

MacroReleases(series_id='UNRATE', name='Unemployment Rate', source='FRED', frequency=<Frequency.MONTHLY: 'M'>, unit='%', country='US', category='Labor', seasonally_adjusted=None)

In [6]:
comparison = pd.DataFrame({
    'first': releases.get_release(1).data,
    'second': releases.get_release(2).data,
    'latest': releases.get_release('latest').data,
    'as_of_2024-04-15': releases.get_release('2024-04-15').data,
})
comparison['revision'] = comparison['latest'] - comparison['first']
comparison

,first,second,latest,as_of_2024-04-15,revision
date,,,,,
2024-01-01,3.7,3.8,3.8,3.8,0.1
2024-02-01,3.9,4.0,4.0,3.9,0.1
2024-03-01,3.8,3.7,3.7,3.8,-0.1
2024-04-01,3.9,4.0,4.0,NaN,0.1


In [7]:
releases.get_release(1)

MacroSeries(series_id='UNRATE', name='Unemployment Rate', source='FRED', frequency=<Frequency.MONTHLY: 'M'>, unit='%', country='US', category='Labor', release=1, seasonally_adjusted=None, mean=3.8249999999999997, std=0.0957427107756337, start=Timestamp('2024-01-01 00:00:00'), end=Timestamp('2024-04-01 00:00:00'), n_obs=4)

In [8]:
bad_releases = {'zero': 0, 'bool': True, 'float': 3.5,
                'unknown_text': 'hola', 'before_any_data': '2020-01-01'}
for label, r in bad_releases.items():
    try:
        releases.get_release(r)
        print(f"{label}: NO ERROR (wrong)")
    except (TypeError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

bad_data = {
    'not_a_dataframe': vintages['value'],
    'missing_column': vintages.drop(columns='release_date'),
    'text_values': vintages.assign(value='x'),
    'bad_dates': vintages.assign(release_date='not a date'),
}
for label, d in bad_data.items():
    try:
        MacroReleases(data=d, **RELEASES_META)
        print(f"{label}: NO ERROR (wrong)")
    except (TypeError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

zero: ValueError: release must be >= 1 (1 = first publication), got 0
bool: TypeError: release must be an int, 'latest' or a date, got bool
float: TypeError: release must be an int, 'latest' or a date, got float
unknown_text: ValueError: release must be an int, 'latest' or a date, got 'hola'
before_any_data: ValueError: release must be on or after the first release_date 2024-02-02, got 2020-01-01
not_a_dataframe: TypeError: data must be a DataFrame, got Series
missing_column: ValueError: data must have columns ['date', 'release_date', 'value'], got ['date', 'value']
text_values: ValueError: data values must be numeric, got dtype str
bad_dates: ValueError: release_date must contain dates, got dtype str


## FREDFetcher: real data
Downloads from FRED (latest values) and ALFRED (all versions) and returns the toolkit's own objects. Needs `FRED_API_KEY` in `.env`.

In [9]:
fetcher = FREDFetcher()
unrate = fetcher.get_macro_series('UNRATE', start='2020-01-01')
unrate

MacroSeries(series_id='UNRATE', name='Unemployment Rate', source='FRED', frequency=<Frequency.MONTHLY: 'M'>, unit='%', country='US', category=None, release='latest', seasonally_adjusted=True, mean=4.776249999999999, std=2.05495657350401, start=Timestamp('2020-01-01 00:00:00'), end=Timestamp('2026-09-01 00:00:00'), n_obs=80)

In [10]:
gdp_releases = fetcher.get_macro_releases('GDP', start='2020-01-01')
print(gdp_releases)

gdp_comparison = pd.DataFrame({
    'first': gdp_releases.get_release(1).data,
    'latest': gdp_releases.get_release('latest').data,
})
gdp_comparison['revision'] = gdp_comparison['latest'] - gdp_comparison['first']
gdp_comparison.tail()

MacroReleases(series_id='GDP', name='Gross Domestic Product', source='FRED', frequency=<Frequency.QUARTERLY: 'Q'>, unit='Bil. of $', country='US', category=None, seasonally_adjusted=True)


,first,latest,revision
date,,,
2025-07-01,31095.089,31180.493,85.404
2025-10-01,31490.070,31462.523,-27.547
2026-01-01,31856.257,31906.274,50.017
2026-04-01,32475.210,32563.030,87.820
2026-07-01,33010.036,33010.036,0.000


### Errors
Requests FRED rejects, or that leave no data, raise a clear error.

In [11]:
bad_fred = {
    'unknown_series': lambda: fetcher.get_macro_series('NOEXISTE'),
    'unknown_series_releases': lambda: fetcher.get_macro_releases('NOEXISTE'),
    'no_data_in_range': lambda: fetcher.get_macro_series('UNRATE', start='2100-01-01'),
    'releases_no_data_in_range': lambda: fetcher.get_macro_releases('GDP', start='2100-01-01'),
    'invalid_api_key': lambda: FREDFetcher(api_key='invalid').get_macro_series('UNRATE'),
}
for label, call in bad_fred.items():
    try:
        call()
        print(f"{label}: NO ERROR (wrong)")
    except (ValueError, ConnectionError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

unknown_series: ValueError: FRED rejected the request for 'NOEXISTE': Bad Request.  The series does not exist.
unknown_series_releases: ValueError: FRED rejected the request for 'NOEXISTE': Bad Request.  The series does not exist.
no_data_in_range: ValueError: start/end must overlap the observations of 'UNRATE', got start=2100-01-01 and end=None
releases_no_data_in_range: ValueError: start/end must overlap the periods 1946-01-01 to 2026-07-01 of 'GDP', got start=2100-01-01 and end=None
invalid_api_key: ValueError: FRED rejected the request for 'UNRATE': Bad Request.  The value for variable api_key is not a 32 character alpha-numeric lower-case string.  Read https://fred.stlouisfed.org/docs/api/api_key.html for more information.


## WorldBankFetcher: data for any country
Downloads annual indicators from the World Bank API for one country and returns a MacroSeries with the toolkit's metadata. No API key needed. The World Bank publishes no vintages, so only current values are available.

In [12]:
wb = WorldBankFetcher()
gdp_es = wb.get_macro_series('NY.GDP.MKTP.KD.ZG', country='ES', start='2010', end='2023')
print(gdp_es)
gdp_es.data.tail()

MacroSeries(series_id='NY.GDP.MKTP.KD.ZG', name='GDP (annual % growth)', source='World Bank', frequency=<Frequency.YEARLY: 'Y'>, unit='%', country='ES', category=None, release='latest', seasonally_adjusted=None, mean=1.1060824913737435, std=4.383490657339809, start=Timestamp('2010-01-01 00:00:00'), end=Timestamp('2023-01-01 00:00:00'), n_obs=14)


2019-01-01     1.961179
2020-01-01   -10.940071
2021-01-01     6.683144
2022-01-01     6.370317
2023-01-01     2.460858
dtype: float64

### Comparing countries
Same indicator for several countries, combined into one table.

In [13]:
growth = pd.DataFrame({
    c: wb.get_macro_series('NY.GDP.MKTP.KD.ZG', country=c, start='2015').data
    for c in ['ES', 'DE', 'FR', 'IT']
})
growth.round(2)

,ES,DE,FR,IT
2015-01-01,4.06,1.66,1.07,0.89
2016-01-01,2.92,2.22,0.86,1.24
2017-01-01,2.90,2.80,2.08,1.60
2018-01-01,2.40,1.14,1.65,0.83
2019-01-01,1.96,0.98,2.03,0.43
2020-01-01,-10.94,-4.13,-7.44,-8.87
2021-01-01,6.68,3.91,6.88,8.93
2022-01-01,6.37,1.81,2.72,4.82
2023-01-01,2.46,-0.87,1.44,0.92
2024-01-01,3.46,-0.50,1.19,0.78


### Errors
Invalid inputs are rejected with a clear message.

In [14]:
QUARTERLY_ID = 'DP.DOD.DECD.CR.BC'

bad_requests = {
    'unknown_country': {'series_id': 'NY.GDP.MKTP.KD.ZG', 'country': 'XX'},
    'unknown_indicator': {'series_id': 'NOEXISTE', 'country': 'ES'},
    'no_data_in_range': {'series_id': 'NY.GDP.MKTP.KD.ZG', 'country': 'ES', 'start': '2100'},
    'quarterly_series': {'series_id': QUARTERLY_ID, 'country': 'ES'},
}
for label, kwargs in bad_requests.items():
    try:
        wb.get_macro_series(**kwargs)
        print(f"{label}: NO ERROR (wrong)")
    except (ValueError, ConnectionError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

unknown_country: ValueError: country must have observations of 'NY.GDP.MKTP.KD.ZG', got 'XX'
unknown_indicator: ValueError: World Bank rejected the request 'country/ES/indicator/NOEXISTE': The provided parameter value is not valid
no_data_in_range: ValueError: start/end must overlap the observations of 'NY.GDP.MKTP.KD.ZG', got start=2100 and end=None
quarterly_series: ValueError: metadata of 'DP.DOD.DECD.CR.BC' must be available (only annual series are supported), got none
